# Trích xuất Image Embeddings cho Đồ án Tốt nghiệp (DATN)

Notebook chạy trên Kaggle (GPU) để:
1. Đọc ảnh sản phẩm đã tải sẵn (từ notebook tải ảnh trước đó, đóng gói thành `downloaded_images.zip` rồi upload lên Kaggle làm dataset ảnh riêng)
2. Dùng mô hình đa phương thức **Jina CLIP v2** (`jinaai/jina-clip-v2`, vision encoder EVA-02) để trích xuất vector đặc trưng ảnh (1024 chiều)
3. Lưu kết quả ra `image_embeddings.npy` + file metadata để dùng cho các bước sau (index FAISS, gợi ý sản phẩm...)

Tự động phát hiện và tận dụng toàn bộ GPU khả dụng trên phiên Kaggle (1 hoặc nhiều GPU), có checkpoint để tiếp tục an toàn nếu phiên bị ngắt giữa chừng.

## 1. Cài đặt thư viện

Kaggle có sẵn `transformers` nhưng bản cài sẵn có bug khi nạp Jina CLIP v2 (so sánh nhầm `str` với `int` lúc sort state_dict), nên mình ghim cứng về bản `5.3.0` đã test chạy ổn.

In [ ]:
!pip install -q --upgrade "transformers==5.3.0" polars pillow tqdm requests pyarrow einops timm

## 2. Import thư viện & kiểm tra GPU

In [ ]:
import os
import numpy as np
import polars as pl
from PIL import Image, ImageFile
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel

ImageFile.LOAD_TRUNCATED_IMAGES = True  # tránh lỗi khi đọc ảnh bị khuyết/hỏng một phần

device = "cuda" if torch.cuda.is_available() else "cpu"
n_gpu = torch.cuda.device_count()
print(f"Using device: {device} | Số GPU khả dụng: {n_gpu}")
for i in range(n_gpu):
    print(f"  - cuda:{i}: {torch.cuda.get_device_name(i)}")

## 3. Cấu hình đường dẫn dữ liệu

Notebook này chạy tiếp theo sau bước tải ảnh: ảnh sản phẩm đã được tải và đóng gói thành `downloaded_images.zip` ở notebook trước, giờ upload lên Kaggle thành **2 dataset** riêng rồi add vào notebook:

1. Dataset gốc (`items.parquet`, ...) → `INPUT_DIR`
2. Dataset ảnh (`downloaded_images.zip` đã giải nén) → `IMAGE_INPUT_DIR`

Có fallback sang `../data` để test nhanh ở local trước khi submit.

In [ ]:
INPUT_DIR = "/kaggle/input/datasets/hoho0111/clothing-shoes-and-jewelry"
IMAGE_INPUT_DIR = "/kaggle/input/clothing-shoes-jewelry-images"  # TODO: đổi theo tên dataset ảnh bạn upload
OUTPUT_DIR = "/kaggle/working"

MODEL_NAME = "jinaai/jina-clip-v2"

items_path = os.path.join(INPUT_DIR, "items.parquet")
if not os.path.exists(items_path):
    # fallback để test nhanh ở local
    items_path = "../data/items.parquet"
    INPUT_DIR = "../data"

IMAGE_DIR = os.path.join(IMAGE_INPUT_DIR, "downloaded_images")
if not os.path.exists(IMAGE_DIR):
    if os.path.exists(IMAGE_INPUT_DIR):
        IMAGE_DIR = IMAGE_INPUT_DIR  # zip giải nén phẳng, không có thư mục con
    else:
        IMAGE_DIR = "../data/downloaded_images"  # fallback test local

print(f"Loading items from: {items_path}")
print(f"Loading images from: {IMAGE_DIR}")

## 4. Đọc dữ liệu sản phẩm

In [ ]:
df_items = pl.read_parquet(items_path)
print(f"Tổng số sản phẩm: {len(df_items)}")
df_items.head(3)

## 5. Xác minh ảnh đã tải sẵn

Ảnh đã được tải về và đóng gói thành zip ở notebook trước rồi upload lên Kaggle làm dataset riêng (`IMAGE_DIR`), nên ở đây không tải lại từ mạng nữa — chỉ đối chiếu nhanh xem có thiếu ảnh nào so với `items.parquet` không.

In [ ]:
item_ids = df_items["item_id"].to_list()
existing_files = set(os.listdir(IMAGE_DIR))
missing_ids = [iid for iid in item_ids if f"{iid}.jpg" not in existing_files]

print(f"Tổng số sản phẩm: {len(item_ids)}")
print(f"Số ảnh có trong dataset: {len(existing_files)}")
print(f"Số ảnh bị thiếu: {len(missing_ids)}")
if missing_ids:
    print("Ví dụ 5 item_id bị thiếu ảnh (sẽ dùng ảnh đen thay thế):", missing_ids[:5])

## 6. Định nghĩa Dataset để nạp ảnh cho model

Kế thừa `torch.utils.data.Dataset`, trả về ảnh PIL kèm `item_id`. Ảnh tải lỗi/thiếu thì thay bằng ảnh đen để không làm crash cả batch.

In [ ]:
class ProductImageDataset(Dataset):
    def __init__(self, df, image_dir):
        self.item_ids = df["item_id"].to_list()
        self.image_dir = image_dir

    def __len__(self):
        return len(self.item_ids)

    def __getitem__(self, idx):
        item_id = self.item_ids[idx]
        file_path = os.path.join(self.image_dir, f"{item_id}.jpg")
        try:
            # Mở thẳng file thay vì os.path.exists trước - đỡ một lượt truy vấn ổ đĩa cho mỗi ảnh
            img = Image.open(file_path).convert("RGB")
        except Exception:
            img = Image.new("RGB", (224, 224), color=0)  # ảnh thiếu/lỗi -> ảnh đen thay thế
        return item_id, img

## 7. Tải mô hình Jina CLIP v2

Bản `transformers` cài sẵn trên Kaggle có 3 lỗi tương thích với Jina CLIP v2, cần vá (monkeypatch) trước khi load model:

1. **`dot_natural_key` so sánh nhầm kiểu dữ liệu** khi sort tên tham số trong state_dict → gây `TypeError: '<' not supported between instances of 'str' and 'int'`.
2. **Thiết bị `meta`**: `transformers` khởi tạo model trên thiết bị "ảo" `meta` trước khi nạp trọng số để load nhanh hơn, nên buffer chưa được tính giá trị thật → phải ép về `cpu`.
3. **Buffer non-persistent bị ghi đè bằng vùng nhớ rác**: sau khi nạp xong, `transformers` vẫn ghi đè các buffer không nằm trong checkpoint (RoPE `freqs_cos/sin`, `inv_freq`...) → phải dựng 1 model tham chiếu rồi copy lại đúng giá trị.

Lỗi thứ 3 nguy hiểm nhất vì không ném exception nào cả — model vẫn chạy nhưng ra vector NaN (hoặc sai lệch âm thầm). Vì vậy cuối phần này luôn có bước "smoke test" để bắt lỗi ngay, trước khi tốn hàng giờ GPU chạy trên toàn bộ dataset.

In [ ]:
def patch_dot_natural_key():
    # Lỗi 1/3: sửa hàm sort dùng khi nạp state_dict, không cho so sánh int với str trực tiếp.
    try:
        import transformers.core_model_loading as _core_model_loading
    except ImportError:
        return False
    if not hasattr(_core_model_loading, "dot_natural_key"):
        return False

    def _safe_dot_natural_key(s):
        return [(0, int(p)) if p.isdigit() else (1, p) for p in s.split(".")]

    _core_model_loading.dot_natural_key = _safe_dot_natural_key
    return True

In [ ]:
import contextlib
import gc
import torch.utils._device as _torch_device_mod
from torch.utils._device import _device_constructors


@contextlib.contextmanager
def meta_init_safe_load():
    # Lỗi 2/3: ép mọi tensor tạo trong lúc khởi tạo model về "cpu" thay vì "meta",
    # để buffer được tính giá trị thật ngay từ đầu.
    orig_call = _torch_device_mod.DeviceContext.__torch_function__

    def patched_call(self, func, types, args=(), kwargs=None):
        kwargs = kwargs or {}
        if self.device.type == "meta" and kwargs.get("device") is None and func in _device_constructors():
            kwargs = dict(kwargs)
            kwargs["device"] = "cpu"
            return func(*args, **kwargs)
        return orig_call(self, func, types, args, kwargs)

    _torch_device_mod.DeviceContext.__torch_function__ = patched_call
    try:
        yield
    finally:
        _torch_device_mod.DeviceContext.__torch_function__ = orig_call

In [ ]:
def _iter_non_persistent_buffers(module, prefix=""):
    for name, buf in module._buffers.items():
        if buf is not None and name in module._non_persistent_buffers_set:
            yield (f"{prefix}.{name}" if prefix else name), buf
    for child_name, child in module.named_children():
        child_prefix = f"{prefix}.{child_name}" if prefix else child_name
        yield from _iter_non_persistent_buffers(child, child_prefix)


def restore_non_persistent_buffers(model):
    # Lỗi 3/3 (quan trọng nhất, đã kiểm chứng thực nghiệm): sau from_pretrained,
    # transformers vẫn ghi đè MỌI buffer non-persistent bằng torch.empty_like()
    # (vùng nhớ rác), bất kể patch ở trên đã tính đúng hay chưa.
    # Cách vá: dựng 1 model tham chiếu bằng constructor thẳng (bỏ qua from_pretrained
    # nên không bị ghi đè), rồi copy buffer từ đó sang model thật.
    targets = list(_iter_non_persistent_buffers(model))
    if not targets:
        return []

    with meta_init_safe_load():
        ref_model = type(model)(model.config)
    ref_lookup = dict(_iter_non_persistent_buffers(ref_model))

    restored = []
    for name, buf in targets:
        ref_buf = ref_lookup.get(name)
        if ref_buf is not None and ref_buf.shape == buf.shape:
            buf.data.copy_(ref_buf.data.to(device=buf.device, dtype=buf.dtype))
            restored.append(name)

    del ref_model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    missing = [n for n, _ in targets if n not in restored]
    if missing:
        raise RuntimeError(f"Không khôi phục được {len(missing)} buffer: {missing[:10]}")
    return restored

Trước khi load model, cần vá thêm 2 lỗi môi trường trên Kaggle (không liên quan Jina CLIP nhưng chặn `import torchvision`/`timm` sau khi nâng cấp `transformers`): thiếu thuộc tính `PIL._typing._Ink`, và trùng đăng ký kernel `register_fake` cho `nms`/`roi_align`. Vá thẳng trong RAM (không cần khởi động lại kernel, toàn bộ ảnh/biến hiện có được giữ nguyên).

In [ ]:
import sys
import os
import types
import typing
import PIL

# 1. Định nghĩa _Ink (Type Hint cho màu sắc) mà một số bản Pillow mới thiếu
InkType = typing.Union[tuple, list, str, int, float, typing.Any]

# 2. Ghi bổ sung vào file vật lý trên ổ đĩa để các lần import sau không bị lỗi
try:
    pil_dir = os.path.dirname(PIL.__file__)
    typing_file = os.path.join(pil_dir, "_typing.py")
    if os.path.exists(typing_file):
        with open(typing_file, "r", encoding="utf-8") as f:
            content = f.read()
        if "_Ink" not in content:
            with open(typing_file, "a", encoding="utf-8") as f:
                f.write("\nimport typing\n_Ink = typing.Any\n")
            print(f"Đã bổ sung _Ink vào file: {typing_file}")
except Exception as e:
    print(f"Bỏ qua ghi file (không bắt buộc): {e}")

# 3. Patch trực tiếp vào RAM hiện tại
try:
    import PIL._typing
    PIL._typing._Ink = InkType
except Exception:
    pass

if "PIL._typing" in sys.modules:
    sys.modules["PIL._typing"]._Ink = InkType
else:
    mod = types.ModuleType("PIL._typing")
    mod._Ink = InkType
    sys.modules["PIL._typing"] = mod

# 4. Xóa cache các module import bị lỗi dở dang để nạp lại sạch sẽ
# (chỉ xóa cache import, không ảnh hưởng biến/ảnh đang có trong RAM)
modules_to_clear = ["PIL.ImageText", "PIL.ImageDraw", "torchvision", "timm"]
for mod_name in list(sys.modules.keys()):
    if any(mod_name == pkg or mod_name.startswith(pkg + ".") for pkg in modules_to_clear):
        del sys.modules[mod_name]

print("Đã vá xong PIL._typing - không cần khởi động lại kernel.")

In [ ]:
import sys
import torch
import torch.library

# 1. Patch FakeImpl.register (xử lý lỗi register_fake của torchvision::nms)
try:
    import torch._library.fake_impl
    if hasattr(torch._library.fake_impl, "FakeImpl"):
        orig_fake_register = torch._library.fake_impl.FakeImpl.register
        def _safe_fake_register(self, func, *args, **kwargs):
            kwargs["allow_override"] = True
            try:
                return orig_fake_register(self, func, *args, **kwargs)
            except RuntimeError as e:
                if "already has" in str(e).lower() or "fake impl registered" in str(e).lower():
                    return None
                raise e
        torch._library.fake_impl.FakeImpl.register = _safe_fake_register
except Exception as e:
    print(f"Lưu ý FakeImpl: {e}")

# 2. Patch torch.library.register_fake decorator
if hasattr(torch.library, "register_fake"):
    orig_register_fake = torch.library.register_fake
    def _safe_register_fake(op_name, fn=None, *args, **kwargs):
        kwargs["allow_override"] = True
        dec = orig_register_fake(op_name, *args, **kwargs)
        if fn is not None:
            try:
                return dec(fn)
            except RuntimeError as e:
                if "already has" in str(e).lower():
                    return fn
                raise e
        def wrapper(func):
            try:
                return dec(func)
            except RuntimeError as e:
                if "already has" in str(e).lower():
                    return func
                raise e
        return wrapper
    torch.library.register_fake = _safe_register_fake

# 3. Patch torch.library.Library.impl (xử lý lỗi roi_align trước đó)
def patch_torch_library():
    targets = [
        getattr(torch.library, "Library", None),
        getattr(torch.library, "_ScopedLibrary", None),
    ]
    for target in targets:
        if target is not None and hasattr(target, "impl"):
            orig_impl = target.impl
            def make_patched(old_impl):
                def patched_impl(self, *args, **kwargs):
                    kwargs["allow_override"] = True
                    try:
                        return old_impl(self, *args, **kwargs)
                    except Exception as e:
                        err_str = str(e).lower()
                        if "already a kernel registered" in err_str or "already has" in err_str:
                            return None
                        if "allow_override" in err_str:
                            kwargs.pop("allow_override", None)
                            try:
                                return old_impl(self, *args, **kwargs)
                            except Exception as inner_e:
                                if "already a kernel registered" in str(inner_e).lower():
                                    return None
                                raise inner_e
                        raise e
                return patched_impl
            target.impl = make_patched(orig_impl)

patch_torch_library()

# 4. Xóa cache dở dang của torchvision & timm để nạp lại sạch sẽ
for mod_name in list(sys.modules.keys()):
    if any(mod_name == pkg or mod_name.startswith(pkg + ".") for pkg in ["torchvision", "timm"]):
        del sys.modules[mod_name]

# 5. Nạp trực tiếp torchvision và timm
import torchvision
import timm

print("Đã nạp lại torchvision và timm thành công.")

Áp dụng các patch, load model, và khôi phục buffer.

In [ ]:
_patched = patch_dot_natural_key()
print(f"patch_dot_natural_key applied: {_patched}")

print(f"Khởi tạo Jina CLIP Model: {MODEL_NAME}...")
with meta_init_safe_load():
    model = AutoModel.from_pretrained(MODEL_NAME, trust_remote_code=True, torch_dtype=torch.float32)
model = model.to(device)
model.eval()
print("Tải mô hình thành công.")

_restored = restore_non_persistent_buffers(model)
print(f"Đã khôi phục {len(_restored)} buffer non-persistent (RoPE, ...).")

**Smoke test:** kiểm tra nhanh 1 ảnh mẫu để chắc chắn model không sinh NaN, trước khi chạy tốn thời gian trên toàn bộ dataset.

In [ ]:
_smoke_img = Image.new("RGB", (224, 224), color=(128, 128, 128))
with torch.no_grad():
    _smoke_emb = model.encode_image([_smoke_img], convert_to_numpy=True, show_progress_bar=False)
_smoke_emb = np.asarray(_smoke_emb, dtype=np.float32)
assert not np.isnan(_smoke_emb).any(), "Model sinh vector NaN ngay ở bước kiểm tra nhanh - dừng lại, không chạy full dataset!"
print(f"Smoke test OK - vector ảnh mẫu không chứa NaN (shape={_smoke_emb.shape}, dtype={_smoke_emb.dtype}).")

## 8. Trích xuất đặc trưng cho toàn bộ dataset

Tự động phát hiện số GPU khả dụng (Kaggle có thể cấp 1 GPU P100 hoặc 2 GPU T4 tùy phiên) và chia đều dữ liệu cho từng GPU chạy song song, thay vì hard-code 2 GPU như bản nháp ban đầu (sẽ crash nếu phiên chỉ có 1 GPU). Có checkpoint theo từng lô để lưu an toàn: nếu notebook bị ngắt giữa chừng (hết giờ Kaggle, mất kết nối...), chạy lại sẽ tự phát hiện phần đã xong và tiếp tục thay vì làm lại từ đầu.

**Cấu hình song song hoá:** tự động phát hiện số GPU, chia đều dữ liệu cho từng thiết bị, và tính số worker tải ảnh dựa trên số CPU thực tế của Kaggle thay vì hard-code.

In [ ]:
import copy
import gc

torch.backends.cudnn.benchmark = True  # input đều 224x224 -> cuDNN chọn kernel nhanh nhất cho size cố định

CHECKPOINT_DIR = os.path.join(OUTPUT_DIR, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

BATCH_SIZE = 64            # 1 batch size duy nhất cho cả DataLoader lẫn model - không tách batch 2 lần như bản nháp cũ
SAVE_EVERY_BATCHES = 20    # ghi checkpoint định kỳ để lỡ notebook bị ngắt cũng không mất hết tiến độ

DEVICES = [f"cuda:{i}" for i in range(torch.cuda.device_count())] or ["cpu"]
NUM_WORKERS = max(1, (os.cpu_count() or 4) // len(DEVICES))  # chia đều số CPU thực tế, không hard-code

print(f"Chạy song song trên {len(DEVICES)} thiết bị: {DEVICES}")
print(f"Mỗi DataLoader dùng {NUM_WORKERS} worker để giải mã ảnh song song.")


def split_df(df, n_parts):
    n = len(df)
    base, rem = divmod(n, n_parts)
    parts, start = [], 0
    for i in range(n_parts):
        size = base + (1 if i < rem else 0)
        parts.append(df.slice(start, size))
        start += size
    return parts


n_total = len(df_items)
df_chunks = split_df(df_items, len(DEVICES))
print(f"Tổng {n_total} ảnh, chia thành {[len(c) for c in df_chunks]} ảnh/thiết bị.")

**Nhân bản model & sanity check:** nhân bản model sang từng GPU (nếu có nhiều hơn 1) để suy luận song song, rồi kiểm tra nhanh từng bản sao trước khi chạy full dataset.

In [ ]:
print("Phân bổ model sang từng thiết bị...")
model.to(DEVICES[0])
device_models = [model]
for dev in DEVICES[1:]:
    device_models.append(copy.deepcopy(model).to(dev).eval())
gc.collect()
torch.cuda.empty_cache()

# Sanity check: chống vector NaN / vector 0 trên từng bản sao trước khi chạy full dataset
_test_imgs = [
    Image.new("RGB", (224, 224), color=(128, 128, 128)),
    Image.new("RGB", (224, 224), color=(255, 0, 0)),
]
for dev, worker_model in zip(DEVICES, device_models):
    with torch.no_grad():
        emb = worker_model.encode_image(_test_imgs, convert_to_numpy=True, show_progress_bar=False)
    emb = np.asarray(emb, dtype=np.float32)
    assert not np.isnan(emb).any(), f"[{dev}] Model sinh vector NaN!"
    assert (np.linalg.norm(emb, axis=1) > 1e-3).all(), f"[{dev}] Vector bị triệt tiêu về 0!"
    print(f"  - {dev}: OK (norm={np.round(np.linalg.norm(emb, axis=1), 4)})")

print("Sanity check thành công trên toàn bộ thiết bị.")

**Hàm xử lý cho từng thiết bị:** đọc ảnh qua `DataLoader` (song song bằng nhiều worker), suy luận theo batch, kiểm tra NaN ngay trên từng batch, và ghi checkpoint an toàn (ghi ra file tạm rồi đổi tên bằng `os.replace` để không bao giờ để lại file `.npz` dở dang nếu bị ngắt giữa chừng). Nếu tìm thấy checkpoint cũ của thiết bị này, tự động bỏ qua phần đã xong và chạy tiếp.

In [ ]:
def existing_progress(gpu_id):
    files = sorted(f for f in os.listdir(CHECKPOINT_DIR) if f.startswith(f"chunk_gpu{gpu_id}_") and f.endswith(".npz"))
    n_images = 0
    for f in files:
        with np.load(os.path.join(CHECKPOINT_DIR, f)) as data:
            n_images += len(data["item_ids"])
    return len(files), n_images


def save_chunk_atomic(gpu_id, chunk_idx, embeddings, item_ids):
    final_path = os.path.join(CHECKPOINT_DIR, f"chunk_gpu{gpu_id}_{chunk_idx:04d}.npz")
    tmp_path = final_path + ".tmp"
    with open(tmp_path, "wb") as fh:
        np.savez_compressed(fh, embeddings=embeddings, item_ids=np.array(item_ids))
    os.replace(tmp_path, final_path)  # ghi nguyên tử -> không bao giờ có file .npz dở dang


def process_device_chunk(gpu_id, device_str, worker_model, df_chunk):
    n_done_files, n_done_images = existing_progress(gpu_id)
    if n_done_images > 0:
        print(f"[{device_str}] Tìm thấy checkpoint cũ: {n_done_images} ảnh đã xong, tiếp tục từ đó...")
        df_chunk = df_chunk.slice(n_done_images, len(df_chunk) - n_done_images)

    dataset = ProductImageDataset(df_chunk, IMAGE_DIR)
    loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=NUM_WORKERS, pin_memory=device_str.startswith("cuda"),
        persistent_workers=True,
        collate_fn=lambda batch: ([x[0] for x in batch], [x[1] for x in batch]),
    )

    chunk_idx = n_done_files
    buf_embs, buf_ids = [], []

    def flush():
        nonlocal chunk_idx
        if not buf_embs:
            return
        save_chunk_atomic(gpu_id, chunk_idx, np.vstack(buf_embs).astype(np.float32), buf_ids)
        buf_embs.clear()
        buf_ids.clear()
        chunk_idx += 1

    with torch.no_grad():
        for batch_idx, (batch_ids, batch_imgs) in enumerate(tqdm(loader, desc=device_str, position=gpu_id, leave=True)):
            with torch.autocast(device_str.split(":")[0], dtype=torch.float16, enabled=device_str.startswith("cuda")):
                features = worker_model.encode_image(batch_imgs, convert_to_numpy=True, show_progress_bar=False)
            features = np.asarray(features, dtype=np.float32)

            nan_mask = np.isnan(features).any(axis=1)
            if nan_mask.any():
                bad_ids = [batch_ids[j] for j in np.where(nan_mask)[0]]
                raise RuntimeError(f"[{device_str}] Phát hiện vector NaN tại item_id: {bad_ids[:5]}")

            buf_embs.append(features)
            buf_ids.extend(batch_ids)
            if (batch_idx + 1) % SAVE_EVERY_BATCHES == 0:
                flush()
        flush()

    return chunk_idx

**Chạy song song và gộp kết quả:** mỗi thiết bị chạy trên 1 luồng điều phối (suy luận GPU tự giải phóng GIL nên không tranh chấp lẫn nhau), sau đó gộp toàn bộ checkpoint lại theo đúng thứ tự ban đầu.

In [ ]:
print(f"Bắt đầu trích xuất {n_total} ảnh trên {len(DEVICES)} thiết bị (checkpoint tại {CHECKPOINT_DIR})...")

with ThreadPoolExecutor(max_workers=len(DEVICES)) as executor:
    futures = [
        executor.submit(process_device_chunk, gpu_id, dev, worker_model, df_chunk)
        for gpu_id, (dev, worker_model, df_chunk) in enumerate(zip(DEVICES, device_models, df_chunks))
    ]
    for f in futures:
        f.result()

print("Đã trích xuất xong, đang gộp checkpoint từ ổ đĩa theo đúng thứ tự...")


def load_device_chunks(gpu_id):
    files = sorted(f for f in os.listdir(CHECKPOINT_DIR) if f.startswith(f"chunk_gpu{gpu_id}_") and f.endswith(".npz"))
    embs, ids = [], []
    for f in files:
        with np.load(os.path.join(CHECKPOINT_DIR, f)) as data:
            embs.append(data["embeddings"])
            ids.extend(data["item_ids"].tolist())
    return np.vstack(embs), ids


all_embs, item_ids_ordered = [], []
for gpu_id in range(len(DEVICES)):
    embs, ids = load_device_chunks(gpu_id)
    all_embs.append(embs)
    item_ids_ordered.extend(ids)

image_embeddings = np.vstack(all_embs).astype(np.float32)
assert len(item_ids_ordered) == n_total, f"Số lượng ({len(item_ids_ordered)}) không khớp tổng ({n_total})!"
print(f"Hoàn tất! Kích thước ma trận embeddings hình ảnh: {image_embeddings.shape}, dtype: {image_embeddings.dtype}")

## 9. Lưu kết quả

Kiểm tra lại NaN/Inf/dtype lần cuối, sau đó ghi ra file `.tmp` rồi đổi tên (`os.replace`) để không bao giờ để lại file `.npy`/`.parquet` dở dang nếu notebook bị ngắt giữa chừng.

In [ ]:
assert not np.isnan(image_embeddings).any(), "Phát hiện NaN trong embeddings trước khi lưu!"
assert not np.isinf(image_embeddings).any(), "Phát hiện Inf trong embeddings trước khi lưu!"
assert image_embeddings.dtype == np.float32, f"Kiểu dữ liệu không mong đợi: {image_embeddings.dtype}"

emb_output_path = os.path.join(OUTPUT_DIR, "image_embeddings.npy")
with open(emb_output_path + ".tmp", "wb") as fh:
    np.save(fh, image_embeddings)
os.replace(emb_output_path + ".tmp", emb_output_path)  # ghi nguyên tử
print(f"Đã lưu ma trận vector nhúng ảnh tại: {emb_output_path}")

df_metadata = pl.DataFrame({
    "index": list(range(len(item_ids_ordered))),
    "item_id": item_ids_ordered,
})
meta_output_path = os.path.join(OUTPUT_DIR, "image_embedding_metadata.parquet")
df_metadata.write_parquet(meta_output_path + ".tmp")
os.replace(meta_output_path + ".tmp", meta_output_path)  # ghi nguyên tử
print(f"Đã lưu metadata hình ảnh tại: {meta_output_path}")

## 10. Kiểm tra lại file đã lưu

In [ ]:
loaded_embeddings = np.load(emb_output_path)
print(f"Kiểm tra kích thước file tải lại: {loaded_embeddings.shape}, dtype: {loaded_embeddings.dtype}")
assert not np.isnan(loaded_embeddings).any(), "File đã lưu chứa NaN!"
assert np.allclose(image_embeddings, loaded_embeddings), "Dữ liệu lưu bị lỗi!"
print("Kiểm tra hoàn tất: không có NaN, dữ liệu khớp với bản gốc trong bộ nhớ.")

## 11. Dọn dẹp checkpoint tạm

Sau khi đã xác nhận `image_embeddings.npy` hợp lệ ở bước trên, xoá thư mục checkpoint để không tốn thêm dung lượng `/kaggle/working` (giới hạn output của Kaggle).

In [ ]:
import shutil

if os.path.exists(CHECKPOINT_DIR):
    shutil.rmtree(CHECKPOINT_DIR)
    print(f"Đã xoá checkpoint tạm tại: {CHECKPOINT_DIR}")